In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ==============================================================================
# KAGGLE SINGLE-CELL: OLLAMA + F16 ABLITERATED MODEL + OPENAI-COMPATIBLE API
# Exposes the local Ollama server via a Cloudflare Quick Tunnel.
# ==============================================================================

import os, time, subprocess, threading, requests, re
from huggingface_hub import list_repo_files, hf_hub_download

# --- 1. Install zstd & Ollama ---
print("â³ Installing system dependencies and Ollama...")
!apt-get update -qq && apt-get install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q huggingface_hub

if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    raise RuntimeError("Ollama installation failed.")

# --- 2. Start Ollama Server (Dual T4) ---
print("â³ Starting Ollama server for 2x T4 GPUs...")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["OLLAMA_SCHED_SPREAD"] = "1"
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

# --- 3. Download & Import Unquantized Abliterated GGUF ---
print("â³ Searching for an unquantized abliterated GGUF...")
REPO_CANDIDATES = [
    "mradermacher/Meta-Llama-3.1-8B-Instruct-abliterated-Sabresooth-GGUF",
    "legraphista/Meta-Llama-3.1-8B-Instruct-abliterated-IMat-GGUF",
]
LOCAL_DIR = "/kaggle/working/models"
os.makedirs(LOCAL_DIR, exist_ok=True)

chosen_repo, chosen_file = None, None
for repo in REPO_CANDIDATES:
    try:
        files = list_repo_files(repo)
        unquant = [f for f in files if f.lower().endswith(".gguf")
                   and ("f16" in f.lower() or "fp16" in f.lower())]
        if unquant:
            chosen_repo, chosen_file = repo, unquant[0]
            print(f"  âœ… Found: {chosen_file} in {repo}")
            break
    except Exception as e:
        print(f"  âš ï¸ Error checking {repo}: {e}")

if not chosen_file:
    raise RuntimeError("No unquantized abliterated GGUF found.")

print(f"â³ Downloading {chosen_file} ...")
local_path = hf_hub_download(
    repo_id=chosen_repo, filename=chosen_file,
    local_dir=LOCAL_DIR, local_dir_use_symlinks=False,
)
print(f"âœ… Downloaded ({round(os.path.getsize(local_path)/1e9, 2)} GB)")

print("â³ Importing model into Ollama...")
mf_path = "/kaggle/working/Modelfile"
with open(mf_path, "w") as f:
    f.write(f"""FROM {local_path}
PARAMETER num_ctx 8192
PARAMETER temperature 0.7
PARAMETER top_p 0.9
""")
MODEL_NAME = "abliterated-f16"
!ollama create {MODEL_NAME} -f {mf_path}
print(f"âœ… Model '{MODEL_NAME}' created.")

# --- 4. Verify local OpenAI endpoint ---
r = requests.get("http://127.0.0.1:11434/v1/models", timeout=10)
print("Local /v1/models status:", r.status_code)

# --- 5. Install & Start Cloudflare Quick Tunnel ---
print("â³ Installing cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("â³ Starting Cloudflare Quick Tunnel...")
# The --http-host-header flag prevents Ollama from returning 403 for tunneled requests.
tunnel_cmd = [
    "cloudflared", "tunnel", "--url", "http://127.0.0.1:11434",
    "--http-host-header", "localhost:11434"
]
tunnel_proc = subprocess.Popen(
    tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:  # wait up to 30s for URL
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    print(line, end="")
    # Look for the trycloudflare.com URL in the output
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    # Fallback: try a broader regex on accumulated output if needed
    raise RuntimeError("Could not extract the Cloudflare Quick Tunnel URL. Check the output above.")

print("\n" + "="*70)
print("âœ… OPENAI-COMPATIBLE API IS LIVE")
print("="*70)
print(f"Base URL : {public_url}/v1")
print(f"API Key  : ollama   (any non-empty string works; Ollama ignores it[reference:4])")
print(f"Model    : {MODEL_NAME}")
print("="*70)
print("\nPython client example for your PC:")
print(f"""
from openai import OpenAI
client = OpenAI(
    base_url="{public_url}/v1",
    api_key="ollama",
)
resp = client.chat.completions.create(
    model="{MODEL_NAME}",
    messages=[{{"role": "user", "content": "Hello"}}],
)
print(resp.choices[0].message.content)
""")

# --- 6. Keep-Alive Loop (pins model & monitors tunnel) ---
print("ðŸ”„ Entering keep-alive loop. The tunnel and model will remain available.")
consecutive_failures = 0
while True:
    try:
        # Health-check local Ollama
        resp = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
        if resp.status_code == 200:
            models = [m["name"] for m in resp.json().get("models", [])]
            print(f"[{time.strftime('%H:%M:%S')}] âœ… Ollama OK Â· {models}")
            consecutive_failures = 0
        else:
            consecutive_failures += 1
            print(f"[{time.strftime('%H:%M:%S')}] âš ï¸ Ollama status {resp.status_code}")

        # Pin the model in VRAM
        try:
            requests.post("http://127.0.0.1:11434/api/generate", json={
                "model": MODEL_NAME, "prompt": "ping", "stream": False, "keep_alive": -1
            }, timeout=30)
        except Exception:
            pass

        # Check if the tunnel process is still alive
        if tunnel_proc.poll() is not None:
            print("ðŸš¨ Tunnel process died. Restarting cloudflared...")
            tunnel_proc = subprocess.Popen(
                tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
            )
            # Attempt to read a new URL (will take a few seconds)
            time.sleep(5)
            # Note: A new URL will be generated; you may need to update your client.

    except requests.exceptions.RequestException as e:
        consecutive_failures += 1
        print(f"[{time.strftime('%H:%M:%S')}] âš ï¸ Heartbeat failed: {e}")

    if consecutive_failures >= 3:
        print("ðŸš¨ Ollama appears dead. Restarting...")
        subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
        time.sleep(3)
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(8)
        subprocess.run(["ollama", "create", MODEL_NAME, "-f", mf_path], capture_output=True)
        consecutive_failures = 0

    time.sleep(300)  # 5 minutes